> **LangChain 1.x / 2026** — *2026** — drug-discovery evidence is auditable and human-gated; optional paid LLM only where noted. See `UPDATE_2026.md`.

# Chapter 8 — Active Learning Planning (v2026) (v2026)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/IvanReznikov/LangChain4LifeSciencesHealthcare/blob/main/notebooks/Chapter%2008.%20LangChain%20for%20Drug%20Discovery/LC4LSH_Chapter_8_Active_Learning_Planning.ipynb)

**Learning objectives**
- Select the next experiments by a declared acquisition rule
- Balance high-uncertainty and diverse candidates
- Require human approval before any experiment is committed
- Record the decision rule and rationale for audit

> Runtime: ~5 min (local, sklearn)  
> Cost: free  
> Data: small built-in candidate pool with model uncertainty

## Environment setup

### Secrets (optional LLM only)

In [ ]:
import os


def get_secret(name, default=None):
    try:
        from google.colab import userdata  # type: ignore
        v = userdata.get(name)
        if v:
            return v
    except Exception:
        pass
    try:
        from dotenv import load_dotenv  # type: ignore
        load_dotenv()
    except Exception:
        pass
    return os.environ.get(name, default)


# These notebooks are LOCAL-first (RDKit/pandas/sklearn); a paid LLM is OPTIONAL.
OPENAI_API_KEY = get_secret("LC4LS_OPENAI_API_KEY", "")
if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
print("OpenAI key set (optional):", bool(OPENAI_API_KEY))

### Install pinned dependencies

In [ ]:
%pip install -q rdkit "pandas>=2.0" "numpy>=1.26" "matplotlib>=3.8" "scikit-learn>=1.4" "langchain==1.0.0" "langchain-openai==1.0.0" "python-dotenv>=1.0" # Pinned versions - Last validated: 2026-07-21 (see UPDATE_2026.md)

In [ ]:
# Optional LangSmith tracing (only if a key is present)
import os
LANGSMITH_API_KEY = get_secret("LANGCHAIN_API_KEY", "") or ""
LANGSMITH_PROJECT = "lc4lsh-chapter8-active-learning"
if LANGSMITH_API_KEY.startswith(("lsv2_", "ls__")):
    os.environ.setdefault("LANGSMITH_TRACING", "true")
    os.environ.setdefault("LANGSMITH_API_KEY", LANGSMITH_API_KEY)
    os.environ.setdefault("LANGSMITH_PROJECT", LANGSMITH_PROJECT)
    print("LangSmith tracing ON ->", LANGSMITH_PROJECT)
else:
    print("LangSmith OFF (no key) - fine; these notebooks are local-first.")

## Active learning with a human in the loop

Active learning chooses **which experiment to run next** to learn the most. But selecting an experiment is a resource decision: it must follow a **declared rule**, balance **uncertainty** and **diversity**, and require **human approval**. This notebook keeps that auditable.

## 1. Candidate pool + a tiny model with uncertainty

In [ ]:
import numpy as np, pandas as pd
from sklearn.ensemble import RandomForestRegressor
rng = np.random.default_rng(11)
n = 24
X = rng.uniform(0, 1, size=(n, 3))  # toy descriptors
# measured labels for a small labelled subset
labelled = rng.choice(n, size=8, replace=False)
y = (2 * X[:, 0] - X[:, 1] + rng.normal(0, 0.05, n))
df = pd.DataFrame(X, columns=["d1", "d2", "d3"])
df["id"] = [f"C{i}" for i in range(n)]
df["measured"] = False; df.loc[labelled, "measured"] = True
df["y"] = np.where(df["measured"], y, np.nan)
print(df.head(6).to_string(index=False), "\nlabelled:", int(df.measured.sum()), "unlabelled:", int((~df.measured).sum()))

## 2. Predict with an uncertainty estimate (ensemble std)

In [ ]:
lab = df[df.measured]; unlab = df[~df.measured]
model = RandomForestRegressor(n_estimators=50, random_state=11).fit(lab[["d1", "d2", "d3"]], lab["y"])
preds = np.stack([t.predict(unlab[["d1", "d2", "d3"]]) for t in model.estimators_])
unlab = unlab.assign(pred=preds.mean(0), unc=preds.std(0).round(3))
print(unlab[["id", "pred", "unc"]].sort_values("unc", ascending=False).head(6).to_string(index=False))

## 3. Declared acquisition rule: uncertainty + diversity

In [ ]:
from sklearn.metrics import pairwise_distances
def acquisition(unlab, already, lam=0.5, k=4):
    # score = uncertainty - lam * (min distance to already-selected) [encourage diversity]
    pts = unlab[["d1", "d2", "d3"]].values
    ref = np.vstack([already, pts[:0]]) if len(already) else np.zeros((1, 3))
    dmin = pairwise_distances(pts, ref).min(1) if len(already) else np.ones(len(pts))
    score = unlab["unc"].values + lam * dmin
    order = np.argsort(-score)[:k]
    return unlab.iloc[order].assign(score=np.round(score[order], 3))

picked = acquisition(unlab, df[df.measured][["d1", "d2", "d3"]].values, lam=0.5, k=4)
print("PROPOSED next experiments (rule: unc + 0.5*diversity):")
print(picked[["id", "pred", "unc", "score"]].to_string(index=False))

## 4. Human approval gate (required)

In [ ]:
HUMAN_APPROVED = False  # <- a person must review and flip this
def commit(picked, approved):
    if not approved:
        print("BLOCKED: no experiments committed. Awaiting human approval of the proposed batch.")
        return []
    print("APPROVED: committing", len(picked), "experiments:", picked["id"].tolist())
    return picked["id"].tolist()

committed = commit(picked, HUMAN_APPROVED)

## 5. Decision record for audit

In [ ]:
import json
record = {
    "rule": "uncertainty + 0.5 * min-distance(diversity)",
    "model": "RandomForestRegressor(50 trees, seed=11)",
    "n_labelled": int(df.measured.sum()),
    "proposed": picked["id"].tolist(),
    "human_approved": HUMAN_APPROVED,
    "committed": committed,
}
print(json.dumps(record, indent=2))

## Limitations & safety notes

- Toy descriptors/model; real active learning needs a validated surrogate with calibrated uncertainty.
- Ensemble std is a heuristic uncertainty proxy, not a calibrated interval.
- The diversity term uses Euclidean distance on toy descriptors; use chemical fingerprints/distance in practice.
- Committing an experiment is a resource/safety decision — the human gate is mandatory. Local/free; no LLM.

In [ ]:
import gc
gc.collect()
for _v in ["records", "df", "model", "llm", "X", "graph"]:
    globals().pop(_v, None)
gc.collect()
print("Cleanup done.")

## Exercises

<details><summary>Why combine uncertainty with diversity?</summary>Pure uncertainty sampling picks redundant near-duplicates; a diversity term spreads experiments across chemical space for more information per assay.</details>

<details><summary>Why require human approval?</summary>Experiments cost resources and may have safety implications; a person must own the decision, not the algorithm.</details>

<details><summary>Why record the acquisition rule?</summary>So the batch is reproducible and auditable — anyone can see why these candidates were chosen.</details>

### Tasks
- **Task A** - Replace ensemble-std with a calibrated uncertainty (e.g., conformal prediction interval width).
- **Task B** - Use Morgan-fingerprint Tanimoto distance for the diversity term instead of Euclidean.
- **Task C** - Implement batch active learning that avoids picking near-duplicates within the same batch.
- **Task D** - Simulate one active-learning round (commit, add labels, refit) and plot predicted uncertainty before/after.